<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# Deep learning - prediction (regression) of house prices

In this notebook, we will train and evaluate a simple 'deep' learning model. The model will be based on a multilayer perceptron neural network (MLP NN) and implemented by using the Google [Tensorflow](https://www.tensorflow.org/) and [Keras](https://keras.io/) libraries.

In [ ]:
# data processing
import numpy as np
import pandas as pd
import random

# sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# other
import warnings
import joblib
warnings.filterwarnings("ignore", category=FutureWarning)

## Auxiliary functions

We prepare some auxiliary functions to simplify different parts of the process.

In [ ]:
# plotting
def plot_results(y_test, y_pred1, y_pred=None, pred_label='Tuned ML model'):

    # prepare data as dataframes
    y_pred1_df = pd.DataFrame(index=y_test.index, data=y_pred1, columns=['Baseline quantity'])
    if y_pred is not None:
        y_pred_df = pd.DataFrame(index=y_test.index, data=y_pred, columns=['Predicted quantity'])

    # calculate errors and prepare as dataframes
    err_df = pd.DataFrame(y_test.values - y_pred1.reshape(-1,1),columns=['Error']); err_df['Origin'] = 'Simple NN model'
    if y_pred is not None:
        model_err_df = pd.DataFrame(y_test.values - y_pred,columns=['Error']); model_err_df['Origin'] = pred_label
        err_df = pd.concat([err_df, model_err_df])

    # residuals plots
    fig, ax = plt.subplots(1,2, figsize=(15, 5))
    # residuals scatter
    sns.regplot(x=y_test, y=y_test, label='Ideal predictions', ax=ax[0])
    sns.regplot(x=y_test, y=y_pred1, label='Simple NN model', ax=ax[0])
    if y_pred is not None:
        sns.regplot(x=y_test, y=y_pred, label=pred_label, ax=ax[0])
    ax[0].set_xlabel('Quantity'); ax[0].set_ylabel('Quantity'); ax[0].legend(); ax[0].grid()

    # residuals distribution plot
    sns.histplot(data=err_df, x='Error', kde=True, hue='Origin', palette=[plt.cm.tab10.colors[1], plt.cm.tab10.colors[2]], bins=30, ax=ax[1])
    ax[1].set_xlabel('Prediction error'); ax[1].legend_.set_title(None); ax[1].grid()

# Neural network model for regression

Let us begin with training the neural network model for the regression task (prediction of house prices).
Again, as seen before, the process can be divided into the usual parts, as follows:

0. Preprocess data

1. Create/build model

2. Tune model

3. Evaluate model

4. Hands on

## 0. (Pre)process data
We will preprocess the data such that it is ready to be plugged into the model. To solve the regression task with neural networks, an important part is the scaling of the input and output. In this way, the model is enabled to initially learn all weighting factors uniformly from all inputs. In the later phase, the model places greater weights on the inputs that are more important to it for quality predictions.

In [ ]:
# location of the data on disk - relative path
data_src = '../data/housing_prices/'

In [ ]:
# Load training data
X_train = pd.read_csv(data_src + 'X_train.csv',index_col='Id')
y_train = pd.read_csv(data_src + 'y_train.csv',index_col='Id')
# Load test data
X_test = pd.read_csv(data_src + 'X_test.csv',index_col='Id')
y_test = pd.read_csv(data_src + 'y_test.csv',index_col='Id')

In [ ]:
# Scale data
ss_input = StandardScaler()
ss_output = StandardScaler()

X_train_scaled = ss_input.fit_transform(X_train)
y_train_scaled = ss_output.fit_transform(y_train)

X_test_scaled = ss_input.transform(X_test)
y_test_scaled = ss_output.transform(y_test)

## 1. Create/Build model
The creation of a neural network model is somewhat more complicated when compared to the basic Sklearn models like linear regression. However, thanks to the high-level Keras library and API, the process can also be simplified.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import regularizers

Before we continue, we have to think about reproducibility of our experiments!

In [ ]:
# fix other seeds
random.seed(321)
np.random.seed(321)

# fix tensorflow seeds
tf.random.set_seed(321)

### Basic way of creating a NN model

The model is build by using a few Keras functions. In general, after the model is initialized ([tf.keras.Sequential](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential)), we continue by iteratively adding additional layers.
Since we are creating a MLP NN model, we add dense, fully connected layers ([tf.keras.layers.Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)) with just basic parameters:
   - number of neurons in the layer,
- activation function choice.

A more comprehensive list of parameters can be found in the Keras API documentation.

In [ ]:
# initialize the NN model (use tf.keras.Sequential)
model = tf.keras.Sequential()

# add input layer to the model (layers.Input+layers.Dense) (3 neurons, input dim 10 and linear activation function)
model.add(layers.Input(shape=(10,)))
model.add(layers.Dense(3, activation='linear'))

# add hidden layers to the model (layers.Dense, 3 neurons)
model.add(layers.Dense(3, activation='relu'))
model.add(layers.Dense(3, activation='relu'))

# output layer (dense output layer with linear activation function)
model.add(layers.Dense(1, activation='linear'))

In [ ]:
# print out the model structure
model.summary()

## 2. Fit model
Once the model is built, we will configure the training procedure using the Keras Model.compile method. The most important arguments to compile are the loss and the optimizer, since these define what will be optimized (mean_absolute_error) and how (using the tf.keras.optimizers.Adam).

In [ ]:
# compile the NN model (MAE, ADAM)
model.compile(loss='mae', optimizer='adam', metrics=['mae'])

# fit the NN model on the dataset (X/y train scaled, 8 batch size, 100 epochs)
history = model.fit(X_train_scaled, y_train_scaled, epochs=100, batch_size=8);

In [ ]:
# plot the values of the loss function during the tuning epochs
plt.figure(figsize=(15, 5))
plt.plot(history.history['loss'], label='training')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid()
plt.show()

## 3. Evaluate model
Now that we have obtained our NN model, we will generate predictions and evaluate the model accuracy.

In [ ]:
# errors dataframe
error = pd.DataFrame(index=['MAE', 'RMSE', 'R2'], columns=['NN simple', 'NN playable'], data=np.NaN)

In [ ]:
# generate predictions
y_pred_scaled = model.predict(X_test_scaled)
# unscale the data
y_pred = ss_output.inverse_transform(y_pred_scaled)
# process the obtained predictions (round to integer, staturate negative values)
y_pred = np.round(y_pred)
y_pred[y_pred < 0] = 0
y_simple = y_pred

In [ ]:
# evaluate model accuracy
nn_mae = mean_absolute_error(y_test, y_pred)
nn_rmse = mean_squared_error(y_test, y_pred, squared=False)
nn_r2 = r2_score(y_test, y_pred)

error['NN simple'] = [nn_mae, nn_rmse, nn_r2]
error

In [ ]:
# visualize the obtained results by using the prepared aux function
plot_results(y_test, y_simple)

### Ready-to-play NN model
To make things more manuverable, we will divide our model build process into two separate functions:
- build_nn_model: part of the code that creates the NN object. As parameters we have used the number of neurons per layer, number of layers, number of model inputs, the number of model outputs, activation function and regularizer.
- tune_nn_model: part of the code that executes the model training procedure. As parameters we have used the scaled X train/test and y train/test datasets. The number of neurons, numver of layers, activation function and regularizer are passed into the build_nn_model function.

Additionally, we will now further separate the training dataset into the training and validation parts, such that the validation part can be utilized to implement the early stopping procedure of NN tuning.

In [ ]:
# form train and validation datasets
X_train_scaled, X_valid_scaled, y_train_scaled, y_valid_scaled = train_test_split(X_train_scaled, y_train_scaled, 
                                                                                  test_size=0.2, random_state=321, shuffle=False)

In [ ]:
def build_nn_model(num_of_neurons, num_of_layers, num_of_inputs, num_of_outputs, activation, regularizer):

    # initialize the model
    model = tf.keras.Sequential()

    # first layer (input and dense input layer with a linear activation function)
    model.add(layers.Input(shape=(num_of_inputs,)))
    model.add(layers.Dense(num_of_neurons, activation='linear'))

    # hidden layers (arbitrary number of layers, neurons, activation function and regularization)
    for layer in range(1, num_of_layers + 1):
        if regularizer == None:
            model.add(layers.Dense(num_of_neurons, activation=activation))
        elif regularizer in ['l1', 'L1']:
            model.add(layers.Dense(num_of_neurons, activation=activation, kernel_regularizer=regularizers.l1()))
        elif regularizer in ['l2', 'L2']:
            model.add(layers.Dense(num_of_neurons, activation=activation, kernel_regularizer=regularizers.l2()))
        elif regularizer in ['l12', 'L12']:
            model.add(layers.Dense(num_of_neurons, activation=activation, kernel_regularizer=regularizers.l12()))
        elif regularizer in ['dropout', 'Dropout']:
            model.add(layers.Dense(num_of_neurons, activation=activation))
            model.add(layers.Dropout(0.1))
        else:
            raise Exception('Regularizer type doesn\'t exist')
            
    # output layer (dense output layer with linear activation function)
    model.add(layers.Dense(num_of_outputs, activation='linear'))

    return model

Additionally, we will implement the early stopping procedure by using the Keras EarlyStopping and ModelCheckpoint callback functions, which will enable us to stop the model training.

We have pre-set the tuning procedure max epochs number to 1000, and the early stopping algorithm patience to 100 epochs. We are monitoring the validation dataset mean absolute error. The batch size is set to 8 samples.

In [ ]:
def tune_nn_model(X_train_scaled, X_valid_scaled, y_train_scaled, y_valid_scaled, 
                  num_of_neurons, num_of_layers, activation, regularizer, optimizer):

    # callbacks for early stopping of the model training procedure
    es = EarlyStopping(monitor='val_mae', mode='min', patience=100)
    mc = ModelCheckpoint('../models/NN_house_prices_tmp.keras', monitor='val_mae', mode='min', save_best_only=True)

    # create NN model (utilize the build_nn_model function)
    model = build_nn_model(num_of_neurons, num_of_layers, 
                           X_train_scaled.shape[1], y_train_scaled.shape[1], 
                           activation, regularizer)

    # compile the NN model
    model.compile(loss='mae', optimizer=optimizer, metrics=['mae'])

    # fit the NN model on the dataset and store the model as well as the training procedure
    history = model.fit(X_train_scaled, y_train_scaled,
                        validation_data=(X_valid_scaled, y_valid_scaled),
                        epochs=1000,
                        batch_size=8,
                        callbacks=[es, mc])

    return model, history

Test different numbers of layers and different numbers of neurons in the layers of the neural network. More input parameters require a more complex network. Test some extremes: 

1) increase the number of neurons in the layers to a very large number and analyze loss function plot, 

2) try setting the parameter num_of_layers to 0 and check what happens to the predictions of the most expensive houses.

Try different activation functions, regularizers and optimizers and explore their parameters. Documentation for these hyperparameters can be found at the following links:

1) activation - https://www.tensorflow.org/api_docs/python/tf/keras/activations

2) regularizer - https://www.tensorflow.org/api_docs/python/tf/keras/regularizers

3) dropout (regularization type) - https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout

4) optimizer - https://www.tensorflow.org/api_docs/python/tf/keras/optimizers

In [ ]:
# fit (train) the NN model
num_of_neurons = 3
num_of_layers = 2
activation = 'relu'  # 'relu', 'sigmoid', 'tanh' ...
regularizer = None  # None, 'l1', 'l2', 'l12' or 'dropout'
optimizer = 'adam'  # 'SGD','RMSprop', 'adam' ...
model, history = tune_nn_model(X_train_scaled, X_valid_scaled, y_train_scaled, y_valid_scaled, 
                               num_of_neurons, num_of_layers, activation, regularizer, optimizer)
model.summary()

In [ ]:
# plot the values of the loss function (on the training and the validation dataset) during the tuning epochs
plt.figure(figsize=(15, 5))
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid()
plt.show()

In [ ]:
# generate predictions
y_pred_scaled = model.predict(X_test_scaled)
# unscale the data
y_pred = ss_output.inverse_transform(y_pred_scaled)
# process the obtained predictions (round to integer, staturate negative values)
y_pred = np.round(y_pred)
y_pred[y_pred < 0] = 0

In [ ]:
# evaluate model accuracy
nnp_mae = mean_absolute_error(y_test, y_pred)
nnp_rmse = mean_squared_error(y_test, y_pred, squared=False)
nnp_r2 = r2_score(y_test, y_pred)

error['NN playable'] = [nnp_mae, nnp_rmse, nnp_r2]
error

In [ ]:
# visualize the obtained results by using the prepared aux function
plot_results(y_test, y_simple, y_pred)

## Hands-on

During the hands on part of the neural network modelling, your task is to tune a NN with the following parameters:
- 10 input dimension and 1 output dimension (linear activation functions for input and output layers),
- 3 hidden layers (relu activation function, L2 regularization with 0.002 as regularization parameter),
- 15 neurons in each hidden layer,
- MAE loss function,
- ADAM optimizer with learning rate 0.003,
- 32 batch size,
- 200 epochs.

With the prepared code block, check the accuracy of the model by implementing different number of neurons, optimizers, batch sizes, training epochs etc.

In [ ]:
# initialize the NN model
model = tf.keras.Sequential()

# add input layer to the model (layers.Input+layers.Dense) (15 neurons, input dim 10 and linear activation function)
model.add()

# add hidden layers to the model (layers.Dense)
model.add()
model.add()
...

# output layer (dense output layer with linear activation function)
model.add()

# compile the NN model (MAE, ADAM)
model.compile()

# fit the NN model on the dataset (X/y train scaled, 32 batch size, 200 epochs)
model.fit()

In [ ]:
# generate predictions and evaluate model accuracy
y_pred_scaled = model.predict(X_test_scaled)
y_pred = ss_output.inverse_transform(y_pred_scaled)
y_pred = np.round(y_pred)
y_pred[y_pred < 0] = 0
nn_mae = mean_absolute_error(y_test, y_pred)
nn_rmse = mean_squared_error(y_test, y_pred, squared=False)
nn_r2 = r2_score(y_test, y_pred)
error['NN_test'] = [nn_mae, nn_rmse, nn_r2]
error

In [ ]:
# visualize the obtained results by using the prepared aux function
plot_results(y_test, y_simple, y_pred)

## Save and load model
Finally, we will see how to save and load the resulting neural network model and the necessary parameters. 

At the beginning, we need to save the neural network model. This implies its architecture (network type, number of layers, layer sizes) and parameters whose values we obtained in training process (weight coefficients between neurons and bias). We save the architecture of the neural network in the file NN_house_prices_final.keras

In addition, to use the saved model we also need information on how to get from the output given by the neural network to the actual prediction value. That's why we need to save information about the scaler. ...

In [ ]:
model.save('../models/NN_house_prices_final.keras')

joblib.dump(ss_input, '../models/NN_house_prices_input_scaler.pkl')
joblib.dump(ss_output, '../models/NN_house_prices_output_scaler.pkl')

Now let's load the saved models.

In [ ]:
model_loaded = tf.keras.models.load_model('../models/NN_house_prices_final.keras')

ss_input_loaded = joblib.load('../models/NN_house_prices_input_scaler.pkl')
ss_output_loaded = joblib.load('../models/NN_house_prices_output_scaler.pkl')

To test the loaded model we will load X_test and y_test again.

In [ ]:
# Load test data
X_test_loaded = pd.read_csv(data_src + 'X_test.csv',index_col='Id')
y_test_loaded = pd.read_csv(data_src + 'y_test.csv',index_col='Id')

# Scale test data
X_test_scaled_loaded = ss_input.transform(X_test_loaded)
y_test_scaled_loaded = ss_output.transform(y_test_loaded)

In [ ]:
# generate predictions
y_pred_scaled_loaded = model_loaded.predict(X_test_scaled_loaded)
# unscale the data
y_pred_loaded = ss_output_loaded.inverse_transform(y_pred_scaled_loaded)
# process the obtained predictions (round to integer, staturate negative values)
y_pred_loaded = np.round(y_pred_loaded)
y_pred_loaded[y_pred < 0] = 0

In [ ]:
# evaluate model accuracy
nnp_mae = mean_absolute_error(y_test, y_pred_loaded)
nnp_rmse = mean_squared_error(y_test, y_pred_loaded, squared=False)
nnp_r2 = r2_score(y_test, y_pred_loaded)

error['NN playable'] = [nnp_mae, nnp_rmse, nnp_r2]
error

In [ ]:
# visualize the obtained results by using the prepared aux function
plot_results(y_test, y_pred, y_pred_loaded)

If the model has been loaded correctly, the green and orange values should match.



_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 4a_NN_House_prices_  
_Instructors: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_  
_Intended user: XY_  
_Date: 1st January 1991_  

_References:_
- _Kaggle House Prices competition: https://www.kaggle.com/c/house-prices-advanced-regression-techniques_
- _Ames Housing Dataset (hosted on Kaggle): https://www.kaggle.com/datasets/prevek18/ames-housing-dataset_
- _Tensorflow Machine Learning platform: https://www.tensorflow.org/_
- _Keras Deep Learning API: https://keras.io/_
- _Tensorflow Sequential NN model: https://www.tensorflow.org/api_docs/python/tf/keras/Sequential_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_

